In [ ]:
#-- Load libraries
library(readr)
library(dplyr)
library(data.table)
library(ggplot2)
library(lubridate)
library(purrr)
library(stringr)
library(tidyverse)
library(openxlsx)
library(nnet)
library(readr)

In [ ]:
#-- Load outcomes
combo <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Combination_Therapy_noBIP_noPsychotic.csv") %>% select(person_id, Case)
aug <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Augmentation_Therapy_noBIP_noPsychotic.csv") %>% select(person_id, Case)
outcome <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_Outcomes_90days_noBIP_noPsychotic.csv") %>%
  mutate(
    FirstDispense.treatment = as.Date(FirstDispense.treatment),
    ExpectedEndDate.treatment = as.Date(ExpectedEndDate.treatment),
    FirstDispense.treatment.episode = as.Date(FirstDispense.treatment.episode),
    ExpectedEndDate.treatment.episode = as.Date(ExpectedEndDate.treatment.episode),
    FirstDispense.other.episode = as.Date(FirstDispense.other.episode),
    ExpectedEndDate.other.episode = as.Date(ExpectedEndDate.other.episode),
    Incidence = as.Date(Incidence)
  ) 

In [ ]:
#== Identify drug-class with at least 2,000 people
classes <- outcome %>%
  group_by(Category.treatment) %>%
  summarise(
    count = n_distinct(person_id)
  ) %>%
  filter(count >= 2000) %>%
  pull(Category.treatment)

#-- filter outcome file for those within one of those classes
outcome_filt <- outcome %>%
    filter(Category.treatment %in% classes)

In [ ]:
#-- Load Phenotypes
pheno <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv")
pheno <- pheno %>% 
    mutate(Income_quantitative =
          case_when(Income == ">200k" ~ 8,
                   Income == "150-200k" ~ 7,
                   Income == "100-150k" ~ 6,
                   Income == "75-100k" ~ 5,
                   Income == "50-75k" ~ 4,
                   Income == "35-50k" ~ 3,
                   Income == "25-35k" ~ 2,
                   Income == "<=25k" ~ 1,
                   TRUE ~ NA_integer_))

first_dispense <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/First_Dispense.csv")
pheno <- pheno %>%
    left_join(first_dispense, by = "person_id")

In [ ]:
#-- Load PGS
pgs <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/PGS/all_traits_PGS.csv")
colnames(pgs)

#-- Read in predicted ancestries
workspace_bucket <- Sys.getenv("WORKSPACE_BUCKET")
anc_file_path <- file.path(workspace_bucket, "ancestry/ancestry_preds.tsv")
anc_df <- read_tsv(pipe(sprintf("gsutil cat %s", anc_file_path)))

anc_df_clean <- anc_df %>%
  mutate(pca_clean = str_remove_all(pca_features, "\\[|\\]")) 
anc_df_clean2 <- anc_df_clean %>%
  separate(pca_clean, into = paste0("PC", 1:16), sep = ",\\s*", convert = TRUE)
pc <- anc_df_clean2 %>% select(research_id, ancestry_pred, PC1:PC16)

pgs_pc <- pgs %>% left_join(pc, by = c("IID" = "research_id"))
eur_pgs_pc <- pgs_pc %>% 
    filter(ancestry_pred == "eur")

# Define all PGS columns to standardize
pgs_cols <- c("ADHD_01_PGS", "Migraine_01_PGS", "Insomnia_01_PGS", 
             "MDD_04_PGS", "SCZ_03_PGS", "BIP_2025_PGS", "BMI_LOO_PGS")

# Standardize each using EUR means and SDs
for (col in pgs_cols) {
  meu <- mean(eur_pgs_pc[[col]], na.rm = TRUE)
  sd <- sd(eur_pgs_pc[[col]], na.rm = TRUE)
  
  new_col <- paste0(col, "_std")
  pgs_pc[[new_col]] <- (pgs_pc[[col]] - meu) / sd
}

In [ ]:
extract_outcomes <- function(class) {
    
    #-- Switchers FROM this class
    switchers <- outcome %>%
      filter(Category.treatment == class,
             FinalClassification.treatment == "Switching") %>%
      group_by(person_id) %>%
      arrange(Incidence, FirstDispense.treatment) %>%
      slice(1) %>%
      ungroup() %>%
      mutate(Outcome = "Switching")
    
    #-- Continuers ON this class: never switched away from it AND clean
    continuers <- outcome %>%
      filter(Category.treatment == class,
             FinalClassification.treatment == "Continuation") %>%
      filter(!person_id %in% switchers$person_id) %>%
      filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
      filter(!person_id %in% combo$person_id[combo$Case == 1]) %>%
      group_by(person_id) %>%
      arrange(Incidence, FirstDispense.treatment) %>%
      slice(1) %>%
      ungroup() %>%
      mutate(Outcome = "Continuation")
  
  #-- Combine: each person has one row per drug class they qualify for
  dat <- bind_rows(switchers, continuers)
  print(table(dat$Outcome))
  
  #-- Join phenotypes and genetics
  input_data <- dat %>%
    left_join(pheno, by = "person_id") %>%
    mutate(age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)) %>%
    left_join(pgs_pc, by = c("person_id" = "IID"))
  
  #-- Continuation as reference
  input_data$Outcome <- factor(input_data$Outcome, levels = c("Continuation", "Switching"))
  input_data$Outcome <- relevel(input_data$Outcome, ref = "Continuation")
  
  return(input_data)
}

In [ ]:
#-- Drug classes to loop over
drug_classes <- c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI")

#-- Run extract_outcomes for each class
results_by_class <- setNames(
  lapply(drug_classes, extract_outcomes),
  drug_classes
)

#-- Inspect counts per class
lapply(results_by_class, function(x) table(x$Outcome))

In [ ]:
#-- Stack all class-specific data frames into one long table
pooled_data <- bind_rows(results_by_class, .id = "DrugClass")

#-- Set reference drug class (SSRI is the natural choice)
pooled_data$DrugClass <- factor(pooled_data$DrugClass, 
                                levels = c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI"))


Polygenic scores were standardised to the EUR-ancestry mean and standard deviation. Analyses were stratified by genetically inferred ancestry (EUR, AFR, AMR), with polygenic scores residualised on 16 global principal components within each ancestry-stratified analytic sample. While the global PCs primarily capture between-ancestry variation, ancestry stratification removes this signal directly; the lower-order PCs index residual within-ancestry stratification (notably admixture variation in AFR and AMR), which the residualisation step controls for.

In [ ]:
glm_interaction_model <- function(input_data, independent) {
  
  print(independent)
  
  library(sandwich)
  library(lmtest)
  library(emmeans)
  
  #-- Filter for complete cases
  input_data <- input_data %>%
    select(person_id, Outcome, DrugClass, age_at_first_AD, sex_at_birth, 
           Income_quantitative, Education_level,
           PC1, PC2, PC3, PC4, PC5, PC6, PC7, PC8, PC9, PC10, 
           PC11, PC12, PC13, PC14, PC15, PC16,
           !!independent) %>%
    filter(complete.cases(.))
  
  if (nrow(input_data) == 0) {
    message("No complete cases remaining")
    return("Incomplete")
  }
  
  #-- Residualise PGS on global PCs
  pc_formula <- as.formula(paste(independent, 
                                 "~ PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10 +
                                  PC11 + PC12 + PC13 + PC14 + PC15 + PC16"))
  input_data$pgs_residualised <- residuals(lm(pc_formula, data = input_data))
  
  #-- Set SSRI as reference
  input_data$DrugClass <- factor(input_data$DrugClass, 
                                 levels = c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI"))
  
  #-- Drop classes with insufficient cases
  outcome_counts <- input_data %>%
    count(DrugClass, Outcome) %>%
    pivot_wider(names_from = Outcome, values_from = n, values_fill = 0)
  
  insufficient <- outcome_counts %>%
    filter(Continuation < 20 | Switching < 20)
  
  if (nrow(insufficient) > 0) {
    message(sprintf("Dropping classes with insufficient cases: %s", 
                    paste(insufficient$DrugClass, collapse = ", ")))
    input_data <- input_data %>% 
      filter(!DrugClass %in% insufficient$DrugClass) %>%
      mutate(DrugClass = droplevels(DrugClass))
  }
  
  n_total <- nrow(input_data)
  n_persons <- length(unique(input_data$person_id))
  
  #-- Fit full and null GLMs
  full_formula <- as.formula(
    "Outcome ~ pgs_residualised * DrugClass + age_at_first_AD + sex_at_birth + 
               Income_quantitative + Education_level"
  )
  null_formula <- as.formula(
    "Outcome ~ pgs_residualised + DrugClass + age_at_first_AD + sex_at_birth + 
               Income_quantitative + Education_level"
  )
  
  mod_full <- tryCatch(
    glm(full_formula, data = input_data, family = binomial),
    error = function(e) {
      message(sprintf("Full model failed for %s: %s", independent, e$message))
      return(NULL)
    }
  )
  
  mod_null <- tryCatch(
    glm(null_formula, data = input_data, family = binomial),
    error = function(e) {
      message(sprintf("Null model failed for %s: %s", independent, e$message))
      return(NULL)
    }
  )
  
  if (is.null(mod_full) || is.null(mod_null)) return("Failed")
  
  #-- Cluster-robust variance for full model
  cluster_var <- input_data$person_id
  V_robust <- vcovCL(mod_full, cluster = ~ cluster_var)
  
  #-- Within-class PGS slopes via emtrends, using cluster-robust vcov
  slopes <- tryCatch(
    emtrends(mod_full, ~ DrugClass, var = "pgs_residualised", vcov. = V_robust),
    error = function(e) {
      message(sprintf("emtrends failed for %s: %s", independent, e$message))
      return(NULL)
    }
  )
  
  if (is.null(slopes)) return("Failed")
  
  slopes_df <- as.data.frame(summary(slopes, infer = c(TRUE, TRUE)))
  
  #-- Pull out interaction-coefficient rows for the contrasts vs SSRI
  robust_se <- coeftest(mod_full, vcov = V_robust)
  robust_ci <- coefci(mod_full, vcov = V_robust)
  
  interaction_rows <- data.frame(
    term       = rownames(robust_se),
    estimate   = robust_se[, "Estimate"],
    std.error  = robust_se[, "Std. Error"],
    statistic  = robust_se[, "z value"],
    p.value    = robust_se[, "Pr(>|z|)"],
    conf.low   = robust_ci[, 1],
    conf.high  = robust_ci[, 2],
    row.names  = NULL
  ) %>%
    filter(grepl(":DrugClass", term)) %>%
    mutate(
      DrugClass = sub(".*DrugClass", "", term),
      type      = "Interaction (vs SSRI)",
      odds_ratio = exp(estimate),
      lower_95   = exp(conf.low),
      upper_95   = exp(conf.high)
    ) %>%
    select(DrugClass, type, estimate, std.error, statistic, p.value,
           odds_ratio, lower_95, upper_95)
  
  #-- Within-class slopes (per-class log-odds change per SD PGS)
  slope_rows <- slopes_df %>%
    rename(estimate = pgs_residualised.trend,
           std.error = SE,
           statistic = z.ratio,
           p.value = p.value,
           conf.low = asymp.LCL,
           conf.high = asymp.UCL) %>%
    mutate(
      type      = "Within-class slope",
      odds_ratio = exp(estimate),
      lower_95   = exp(conf.low),
      upper_95   = exp(conf.high)
    ) %>%
    select(DrugClass, type, estimate, std.error, statistic, p.value,
           odds_ratio, lower_95, upper_95)
  
  #-- Joint Wald test
  joint_test <- tryCatch(
    waldtest(mod_null, mod_full, 
             vcov = function(x) vcovCL(x, cluster = ~ cluster_var),
             test = "Chisq"),
    error = function(e) {
      message(sprintf("Joint test failed for %s: %s", independent, e$message))
      return(NULL)
    }
  )
  
  joint_row <- if (!is.null(joint_test)) {
    data.frame(
      DrugClass = NA_character_,
      type = "Joint_Interaction_Test",
      estimate = NA_real_, std.error = NA_real_,
      statistic = joint_test$Chisq[2],
      p.value = joint_test$`Pr(>Chisq)`[2],
      odds_ratio = NA_real_, lower_95 = NA_real_, upper_95 = NA_real_
    )
  } else NULL
  
  #-- Combine into one tidy output
  tidied <- bind_rows(slope_rows, interaction_rows, joint_row) %>%
    mutate(PGS = independent,
           N_Total = n_total,
           N_Persons = n_persons) %>%
    select(PGS, type, DrugClass, estimate, std.error, statistic, p.value,
           odds_ratio, lower_95, upper_95, N_Total, N_Persons)
  
  return(tidied)
}

In [ ]:
#-- Setup
population_groups <- c("afr", "eur", "amr")
clinical_groups   <- c("Full", "MDD", "noMDD", "ANX", "CIDI-MDD", "CIDI-noMDD")
pgs_list <- c("ADHD_01_PGS_std", "Migraine_01_PGS_std", "Insomnia_01_PGS_std",
              "MDD_04_PGS_std", "SCZ_03_PGS_std", 
              "BIP_2025_PGS_std", "BMI_LOO_PGS_std")

all_list <- list()

# ============================================================================
# Outer loop: clinical groups
# ============================================================================

for (clinical in clinical_groups) {
  
  cat("Clinical:", clinical, "\n")
  
  #-- Filter for clinical group of interest
  if (clinical == "MDD") {
    clinical_data <- pooled_data %>% filter(Depression == 1)
  } else if (clinical == "noMDD") {
    clinical_data <- pooled_data %>% filter(Depression == 0)
  } else if (clinical == "ANX") {
    clinical_data <- pooled_data %>% filter(Anxiety == 1)
  } else if (clinical == "CIDI-MDD") {
    clinical_data <- pooled_data %>% filter(MDD_Incl == 1)
  } else if (clinical == "CIDI-noMDD") {
    clinical_data <- pooled_data %>% filter(MDD_Incl == 0)
  } else {
    clinical_data <- pooled_data
  }
  
  pop_results_list <- list()
  
  # ==========================================================================
  # Inner loop: ancestry groups
  # ==========================================================================
  
  for (population in population_groups) {
    
    cat("  Ancestry:", population, "\n")
    
    #-- Subset to ancestry group
    input_data_anc <- clinical_data %>%
      filter(ancestry_pred == population)
    
    #-- Skip if too few people overall
    if (nrow(input_data_anc) < 100) {
      message(sprintf("  Skipping %s/%s: only %d rows", 
                      clinical, population, nrow(input_data_anc)))
      next
    }
    
    # ========================================================================
    # Innermost loop: PGS
    # ========================================================================
    
    pgs_results_list <- lapply(pgs_list, function(pgs) {
      result <- tryCatch(
        glm_interaction_model(input_data_anc, pgs),
        error = function(e) {
          message(sprintf("    Error for %s in %s/%s: %s", 
                          pgs, clinical, population, e$message))
          return("Error")
        }
      )
      
      if (!is.data.frame(result)) return(NULL)
      
      result %>%
        mutate(
          ClinicalGroup = clinical,
          Population    = population
        )
    })
    
    #-- Drop NULLs and combine
    pgs_results_list <- pgs_results_list[!sapply(pgs_results_list, is.null)]
    
    if (length(pgs_results_list) > 0) {
      pop_results_list[[population]] <- bind_rows(pgs_results_list)
    }
  }
  
  #-- Combine across populations within this clinical group
  if (length(pop_results_list) > 0) {
    all_list[[clinical]] <- bind_rows(pop_results_list)
  }
}

# ============================================================================
# Combine into single final results table
# ============================================================================

final_results <- bind_rows(all_list)

# ============================================================================
# Multiple testing correction within each clinical x population x type stratum
# ============================================================================

final_results <- final_results %>%
  group_by(DrugClass, ClinicalGroup, Population, type) %>%
  mutate(
    Bonf_P = pmin(p.value * length(pgs_list), 1),
    FDR_P  = p.adjust(p.value, method = "BH")
  ) %>%
  ungroup() %>%
  mutate(
    p_value_display = case_when(
      p.value < 2.2e-16 ~ "<2.2e-16",
      TRUE ~ formatC(p.value, format = "e", digits = 2)
    ),
    Bonf_P_display = case_when(
      Bonf_P < 2.2e-16 ~ "<2.2e-16",
      TRUE ~ formatC(Bonf_P, format = "e", digits = 2)
    )
  )

In [ ]:
final_results_formatted <- final_results %>%
    rename(GeneticMarker = PGS) %>%
    mutate(
        GeneticMarker = case_when(
        GeneticMarker == "ADHD_01_PGS_std" ~ "ADHD",
        GeneticMarker == "Migraine_01_PGS_std" ~ "Migraine",
        GeneticMarker == "Insomnia_01_PGS_std" ~ "Insomnia",
        GeneticMarker == "ANX_2025_PGS_std" ~ "Anxiety",
        GeneticMarker == "MDD_04_PGS_std" ~ "Depression",
        GeneticMarker == "SCZ_03_PGS_std" ~ "Schizophrenia",
        GeneticMarker == "BIP_2025_PGS_std" ~ "Bipolar Disorder",
        GeneticMarker == "BMI_LOO_PGS_std" ~ "BMI",
        TRUE ~ NA_character_)) %>%
    mutate(Population = case_when(
        Population == "afr" ~ "African",
        Population == "eur" ~ "European",
        Population == "amr" ~ "Latino/Mixed-American",
        TRUE ~ NA_character_)) %>%
    mutate(ClinicalGroup = case_when(
        ClinicalGroup == "Full" ~ "Full Cohort",
        ClinicalGroup == "MDD" ~ "Recorded Depression",
        ClinicalGroup == "noMDD" ~ "No Recorded Depression",
        ClinicalGroup == "ANX" ~ "Recorded Anxiety",
        TRUE ~ ClinicalGroup))

pop_order <- c("European", "African", "Latino/Mixed-American")
clinical_order <- c(
  "Full Cohort",
  "Recorded Depression",
  "No Recorded Depression",
  "Recorded Anxiety",
  "CIDI-MDD",
  "CIDI-noMDD"
)

final_results_formatted <- final_results_formatted %>%
  rename(PGS = GeneticMarker) %>%
  mutate(
    Population = factor(Population, levels = pop_order),
    ClinicalGroup = factor(ClinicalGroup, levels = clinical_order)
  ) %>%
  arrange(Population, ClinicalGroup)


Within-class polygenic-score slopes were computed from the full PGS × DrugClass interaction model using emmeans::emtrends, with cluster-robust covariance (sandwich::vcovCL) accounting for individuals contributing to multiple drug classes. Joint tests of effect modification across drug classes were performed via Wald tests (lmtest::waldtest) comparing the full model to a null model excluding all interaction terms, again using cluster-robust variance.

In [ ]:
wb <- loadWorkbook("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx")
removeWorksheet(wb, "Table6")
addWorksheet(wb, "Table6")
writeData(wb, "Table6", final_results_formatted)
saveWorkbook(wb, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", overwrite=TRUE)

In [ ]:
library(dplyr)
library(ggplot2)
library(stringr)
library(cowplot)
library(readxl)

#-- Set path
path <- "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures"

#-- read in results
final_results <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", sheet = "Table6") 
final_results <- final_results %>%
    filter(Population == "European" & ClinicalGroup == "Recorded Depression") %>%
    mutate(PGS = ifelse(PGS =="Bipolar Disorder", "Bipolar", PGS))

# ============================================================================
# Setup: labels, colours, ordering, significance threshold
# ============================================================================

#-- Bonferroni threshold across PGS
n_pgs <- 7
bonf_threshold <- 0.05 / n_pgs   # = 0.00625

#-- Drug class colours (Okabe-Ito, colourblind-safe)
drug_colors <- c(
  "SSRI"  = "#0072B2",
  "SNRI"  = "#56B4E9",
  "TCA"   = "#D55E00",
  "NaSSA" = "#E69F00",
  "SARI"  = "#009E73",
  "NDRI"  = "#CC79A7"
)

#-- Drug class order
drug_order <- c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI")

#-- PGS display order (psychiatric first, then somatic)
pgs_order <- c("Depression", "ADHD",
               "Bipolar", "Schizophrenia", "Insomnia",
               "Migraine", "BMI")


# ============================================================================
# PLOT 1: Within-class PGS slopes, faceted by PGS
# ============================================================================

within_class <- final_results %>%
  filter(type == "Within-class slope") %>%
  mutate(
    DrugClass = factor(DrugClass, levels = drug_order),
    PGS_label = factor(PGS, levels = pgs_order),
    sig_bonf  = p.value < bonf_threshold,
    # Fill = class colour if significant, white if not
    fill_col  = ifelse(sig_bonf, as.character(DrugClass), "ns")
  )

#-- Build a fill palette: class colours for sig, white for non-sig
fill_palette <- c(drug_colors, "ns" = "white")

p_slopes <- within_class %>%
  ggplot(aes(x = odds_ratio, y = DrugClass, colour = DrugClass)) +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95),
                 height = 0.25, linewidth = 0.5) +
  geom_point(aes(fill = fill_col), shape = 23, size = 3, stroke = 0.7) +
  scale_colour_manual(values = drug_colors, guide = "none") +
  scale_fill_manual(values = fill_palette, guide = "none") +
  scale_y_discrete(limits = rev(drug_order)) +
  facet_wrap(~ PGS_label, ncol = 4, scales = "free_x") +
  labs(x = "Odds ratio per SD PGS (95% CI)",
       y = NULL,
       title = "PGS effect on switching, within drug class",
       subtitle = sprintf("Filled diamonds: P < %.4f (Bonferroni, %d PGS)",
                          bonf_threshold, n_pgs)) +
  theme_bw(base_size = 13) +
  theme(panel.grid.minor = element_blank(),
        plot.title = element_text(face = "bold"),
        axis.text = element_text(colour = "black"),
        strip.background = element_rect(fill = "grey90"),
        strip.text = element_text(face = "bold"))


# ============================================================================
# PLOT 2: Interaction terms (effect modification vs SSRI baseline)
# ============================================================================

interactions <- final_results %>%
  filter(type == "Interaction (vs SSRI)") %>%
  mutate(
    DrugClass = factor(DrugClass, levels = drug_order),
    PGS_label = factor(PGS, levels = pgs_order),
    sig_bonf  = p.value < bonf_threshold,
    fill_col  = ifelse(sig_bonf, as.character(DrugClass), "ns")
  )

p_interactions <- interactions %>%
  ggplot(aes(x = odds_ratio, y = PGS_label, colour = DrugClass)) +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95),
                 height = 0.3, linewidth = 0.5,
                 position = position_dodge(width = 0.7)) +
  geom_point(aes(fill = fill_col), shape = 23, size = 2.8, stroke = 0.7,
             position = position_dodge(width = 0.7)) +
  scale_colour_manual(values = drug_colors[drug_order != "SSRI"],
                      name = "Drug class\n(vs SSRI)") +
  scale_fill_manual(values = fill_palette, guide = "none") +
  scale_y_discrete(limits = rev(pgs_order)) +
  labs(x = "Interaction OR (95% CI)",
       y = NULL,
       title = "PGS x Drug class interactions",
       subtitle = sprintf("Filled diamonds: P < %.4f (Bonferroni, %d PGS)",
                          bonf_threshold, n_pgs)) +
  theme_bw(base_size = 13) +
  theme(panel.grid.minor = element_blank(),
        plot.title = element_text(face = "bold"),
        axis.text = element_text(colour = "black"),
        legend.position = "bottom")


# ============================================================================
# PLOT 3: Joint Wald test summary across PGS
# ============================================================================

joint_summary <- final_results %>%
  filter(type == "Joint_Interaction_Test") %>%
  mutate(
    PGS_label = factor(PGS, levels = pgs_order),
    sig_bonf  = p.value < bonf_threshold,
    nlogp     = -log10(p.value)
  )

p_joint <- joint_summary %>%
  ggplot(aes(x = nlogp, y = PGS_label, fill = sig_bonf)) +
  geom_col(width = 0.7) +
  geom_vline(xintercept = -log10(bonf_threshold),
             linetype = "dashed", colour = "grey50") +
  scale_fill_manual(values = c("TRUE" = "#D55E00", "FALSE" = "grey70"),
                    name = sprintf("P < %.4f", bonf_threshold)) +
  scale_y_discrete(limits = rev(pgs_order)) +
  labs(x = expression(-log[10]("omnibus P")),
       y = NULL,
       title = "Joint test:\nPGS x Drug class interaction",
       subtitle = sprintf("Wald test, cluster-robust;\ndashed line at Bonferroni P = %.4f",
                          bonf_threshold)) +
  theme_bw(base_size = 13) +
  theme(panel.grid.minor = element_blank(),
        plot.title = element_text(face = "bold"),
        axis.text = element_text(colour = "black"),
        legend.position = "bottom")


# ============================================================================
# PLOT 4: Heatmap of within-class slopes (Bonferroni asterisk only)
# ============================================================================

p_heatmap <- within_class %>%
  mutate(sig_label = ifelse(p.value < bonf_threshold, "*", "")) %>%
  ggplot(aes(x = DrugClass, y = PGS, fill = log(odds_ratio))) +
  geom_tile(colour = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("%.2f%s", odds_ratio, sig_label)),
            size = 3.5, colour = "black") +
  scale_x_discrete(limits = drug_order) +
  scale_y_discrete(limits = rev(pgs_order)) +
  scale_fill_gradient2(low = "#0072B2", mid = "white", high = "#D55E00",
                       midpoint = 0, name = "log(OR)") +
  labs(x = NULL, y = NULL,
       title = "Within-class PGS effects on switching",
       subtitle = sprintf("Cell values: OR per SD PGS;  *P < %.4f (Bonferroni, %d PGS)",
                          bonf_threshold, n_pgs)) +
  theme_minimal(base_size = 13) +
  theme(panel.grid = element_blank(),
        plot.title = element_text(face = "bold"),
        axis.text = element_text(colour = "black"),
        axis.text.x = element_text(angle = 45, hjust = 1))


# ============================================================================
# Combine and save
# ============================================================================

#-- Main figure: faceted slopes
#ggsave(file.path(path, "PGS_within_class_slopes_Full.png"),
#       plot = p_slopes, device = "png",
#       width = 280, height = 240, units = "mm", dpi = 300)

#-- Two-panel figure: heatmap + joint tests (alternative compact summary)
summary_fig <- plot_grid(p_heatmap, p_joint, ncol = 2,
                         rel_widths = c(1.4, 1),
                         labels = c("A", "B"),
                         align = "h", axis = "tb")

#ggsave(file.path(path, "PGS_DrugClass_summary_Full.png"),
#       plot = summary_fig, device = "png",
#       width = 340, height = 180, units = "mm", dpi = 300)

#-- Three-panel comprehensive figure
top_row <- plot_grid(p_heatmap, p_joint, ncol = 2, rel_widths = c(1.4, 1),
                     labels = c("A", "B"))
full_fig <- plot_grid(top_row, p_slopes, ncol = 1, rel_heights = c(1, 1.2),
                      labels = c("", "C"))

ggsave(file.path(path, "Supp_PGS_DrugClass_Depression.png"),
       plot = full_fig, device = "png",
       width = 270, height = 200, units = "mm", dpi = 300)

#-- Interaction plot (supplementary)
#ggsave(file.path(path, "PGS_DrugClass_interactions_Full.png"),
#       plot = p_interactions, device = "png",
#       width = 200, height = 200, units = "mm", dpi = 300)

#-- View
full_fig

In [ ]:
library(dplyr)
library(ggplot2)
library(stringr)
library(cowplot)
library(readxl)

#-- Set path
path <- "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures"

#-- read in results
final_results <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL_noANX.xlsx", sheet = "Table3b") 
final_results <- final_results %>%
    filter(Population == "European" & ClinicalGroup == "Full Cohort") %>%
    mutate(PGS = ifelse(PGS =="Bipolar Disorder", "Bipolar", PGS))

# ============================================================================
# Setup: labels, colours, ordering, significance threshold
# ============================================================================

#-- Bonferroni threshold across PGS
n_pgs <- 7
bonf_threshold <- 0.05 / n_pgs 

pgs_labels <- c("Depression", "ADHD",
               "Bipolar", "Schizophrenia", "Insomnia",
               "Migraine", "BMI")

#-- Drug class colours (Okabe-Ito, colourblind-safe)
drug_colors <- c(
  "SSRI"  = "#0072B2",
  "SNRI"  = "#56B4E9",
  "TCA"   = "#D55E00",
  "NaSSA" = "#E69F00",
  "NDRI"  = "#CC79A7"
)

#-- Drug class order
drug_order <- c("SSRI", "SNRI", "TCA", "NaSSA", "NDRI")

#-- PGS display order (psychiatric first, then somatic)
pgs_order <- c("Depression", "ADHD",
               "Bipolar", "Schizophrenia", "Insomnia",
               "Migraine", "BMI")

#-- Add PGS_label to results
final_results <- final_results %>%
  mutate(PGS_label = pgs_labels[PGS])


# ============================================================================
# PLOT 1: Within-class PGS slopes, faceted by PGS
# ============================================================================

within_class <- final_results %>%
  filter(type == "Within-class slope") %>%
  mutate(
    DrugClass = factor(DrugClass, levels = drug_order),
    PGS_label = factor(PGS, levels = pgs_order),
    sig_bonf  = p.value < bonf_threshold,
    # Fill = class colour if significant, white if not
    fill_col  = ifelse(sig_bonf, as.character(DrugClass), "ns")
  )

#-- Build a fill palette: class colours for sig, white for non-sig
fill_palette <- c(drug_colors, "ns" = "white")

p_slopes <- within_class %>%
  ggplot(aes(x = odds_ratio, y = DrugClass, colour = DrugClass)) +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95),
                 height = 0.25, linewidth = 0.5) +
  geom_point(aes(fill = fill_col), shape = 23, size = 3, stroke = 0.7) +
  scale_colour_manual(values = drug_colors, guide = "none") +
  scale_fill_manual(values = fill_palette, guide = "none") +
  scale_y_discrete(limits = rev(drug_order)) +
  facet_wrap(~ PGS_label, ncol = 4, scales = "free_x") +
  labs(x = "Odds ratio per SD PGS (95% CI)",
       y = NULL,
       title = "PGS effect on switching, within drug class",
       subtitle = sprintf("Filled diamonds: P < %.4f (Bonferroni, %d PGS)",
                          bonf_threshold, n_pgs)) +
  theme_bw(base_size = 13) +
  theme(panel.grid.minor = element_blank(),
        plot.title = element_text(face = "bold"),
        axis.text = element_text(colour = "black"),
        strip.background = element_rect(fill = "grey90"),
        strip.text = element_text(face = "bold"))


# ============================================================================
# PLOT 2: Interaction terms (effect modification vs SSRI baseline)
# ============================================================================

interactions <- final_results %>%
  filter(type == "Interaction (vs SSRI)") %>%
  mutate(
    DrugClass = factor(DrugClass, levels = drug_order),
    PGS_label = factor(PGS, levels = pgs_order),
    sig_bonf  = p.value < bonf_threshold,
    fill_col  = ifelse(sig_bonf, as.character(DrugClass), "ns")
  )

p_interactions <- interactions %>%
  ggplot(aes(x = odds_ratio, y = PGS_label, colour = DrugClass)) +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95),
                 height = 0.3, linewidth = 0.5,
                 position = position_dodge(width = 0.7)) +
  geom_point(aes(fill = fill_col), shape = 23, size = 2.8, stroke = 0.7,
             position = position_dodge(width = 0.7)) +
  scale_colour_manual(values = drug_colors[drug_order != "SSRI"],
                      name = "Drug class\n(vs SSRI)") +
  scale_fill_manual(values = fill_palette, guide = "none") +
  scale_y_discrete(limits = rev(pgs_order)) +
  labs(x = "Interaction OR (95% CI)",
       y = NULL,
       title = "PGS x Drug class interactions",
       subtitle = sprintf("Filled diamonds: P < %.4f (Bonferroni, %d PGS)",
                          bonf_threshold, n_pgs)) +
  theme_bw(base_size = 13) +
  theme(panel.grid.minor = element_blank(),
        plot.title = element_text(face = "bold"),
        axis.text = element_text(colour = "black"),
        legend.position = "bottom")


# ============================================================================
# PLOT 3: Joint Wald test summary across PGS
# ============================================================================

joint_summary <- final_results %>%
  filter(type == "Joint_Interaction_Test") %>%
  mutate(
    PGS_label = factor(PGS, levels = pgs_order),
    sig_bonf  = p.value < bonf_threshold,
    nlogp     = -log10(p.value)
  )

p_joint <- joint_summary %>%
  ggplot(aes(x = nlogp, y = PGS_label, fill = sig_bonf)) +
  geom_col(width = 0.7) +
  geom_vline(xintercept = -log10(bonf_threshold),
             linetype = "dashed", colour = "grey50") +
  scale_fill_manual(values = c("TRUE" = "#D55E00", "FALSE" = "grey70"),
                    name = sprintf("P < %.4f", bonf_threshold)) +
  scale_y_discrete(limits = rev(pgs_order)) +
  labs(x = expression(-log[10]("omnibus P")),
       y = NULL,
       title = "Joint test:\nPGS x Drug class interaction",
       subtitle = sprintf("Wald test, cluster-robust;\ndashed line at Bonferroni P = %.4f",
                          bonf_threshold)) +
  theme_bw(base_size = 13) +
  theme(panel.grid.minor = element_blank(),
        plot.title = element_text(face = "bold"),
        axis.text = element_text(colour = "black"),
        legend.position = "bottom")


# ============================================================================
# PLOT 4: Heatmap of within-class slopes (Bonferroni asterisk only)
# ============================================================================

p_heatmap <- within_class %>%
  mutate(sig_label = ifelse(p.value < bonf_threshold, "*", "")) %>%
  ggplot(aes(x = DrugClass, y = PGS_label, fill = log(odds_ratio))) +
  geom_tile(colour = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("%.2f%s", odds_ratio, sig_label)),
            size = 3.5, colour = "black") +
  scale_x_discrete(limits = drug_order) +
  scale_y_discrete(limits = rev(pgs_order)) +
  scale_fill_gradient2(low = "#0072B2", mid = "white", high = "#D55E00",
                       midpoint = 0, name = "log(OR)") +
  labs(x = NULL, y = NULL,
       title = "Within-class PGS effects on switching",
       subtitle = sprintf("Cell values: OR per SD PGS;  *P < %.4f (Bonferroni, %d PGS)",
                          bonf_threshold, n_pgs)) +
  theme_minimal(base_size = 13) +
  theme(panel.grid = element_blank(),
        plot.title = element_text(face = "bold"),
        axis.text = element_text(colour = "black"),
        axis.text.x = element_text(angle = 45, hjust = 1))


# ============================================================================
# Combine and save
# ============================================================================

#-- Main figure: faceted slopes
#ggsave(file.path(path, "PGS_within_class_slopes_Full_LL.png"),
#       plot = p_slopes, device = "png",
#       width = 280, height = 240, units = "mm", dpi = 300)

#-- Two-panel figure: heatmap + joint tests (alternative compact summary)
#summary_fig <- plot_grid(p_heatmap, p_joint, ncol = 2,
#                         rel_widths = c(1.4, 1),
#                         labels = c("A", "B"),
#                         align = "h", axis = "tb")

#ggsave(file.path(path, "PGS_DrugClass_summary_Full_LL.png"),
#       plot = summary_fig, device = "png",
#       width = 340, height = 180, units = "mm", dpi = 300)

#-- Three-panel comprehensive figure
top_row <- plot_grid(p_heatmap, p_joint, ncol = 2, rel_widths = c(1.4, 1),
                     labels = c("A", "B"))
full_fig <- plot_grid(top_row, p_slopes, ncol = 1, rel_heights = c(1, 1.2),
                      labels = c("", "C"))

ggsave(file.path(path, "Supp_PGS_DrugClass_Full_LL.png"),
       plot = full_fig, device = "png",
       width = 270, height = 200, units = "mm", dpi = 300)

#-- Interaction plot (supplementary)
#ggsave(file.path(path, "PGS_DrugClass_interactions_Full_LL.png"),
#       plot = p_interactions, device = "png",
#       width = 200, height = 200, units = "mm", dpi = 300)

#-- View
full_fig